# 1. Konfigurasi & Load Environment Variables

In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

# 2. Membaca Semua File `.md` dari Folder

In [2]:
MD_FOLDER_PATH     = "/kaggle/input/datasets/brianmaxwellketaren/panduan-skripsi-rag-md"                # 📁 Folder berisi file .md
COLLECTION_NAME    = "Thesis Guide for USU Computer Science Students"              # Nama collection Qdrant
EMBEDDING_MODEL    = "paraphrase-multilingual-MiniLM-L12-v2"
VECTOR_DIM         = 384                     # Dimensi vektor MiniLM-L12-v2

# Semantic chunking
WINDOW_SIZE        = 2
MIN_SENTENCES      = 2 
PERCENTILE_THRESH  = 80                      # breakpoint_threshold (percentile)

# Retrieval
TOP_K              = 25                      # Jumlah vektor yang di-retrieve
TOP_N_RERANK       = 10                       # Jumlah hasil setelah reranking

# NVIDIA
NVIDIA_MODEL = user_secrets.get_secret("NVIDIA_MODEL")
NVIDIA_API_KEY = user_secrets.get_secret("NVIDIA_API_KEY")
NVIDIA_BASE_URL = user_secrets.get_secret("NVIDIA_BASE_URL")                                    # (~375 token; 5 chunk × 1500 ≈ 1875 token context)
MAX_TOKENS_OUTPUT   = 2048                     # Maks token jawaban LLM
MAX_CHARS_PER_CHUNK = 1500   

HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

# Credentials dari .env
QDRANT_URL         = user_secrets.get_secret("QDRANT_URL")
QDRANT_API_KEY     = user_secrets.get_secret("QDRANT_API_KEY")

# Validasi
assert QDRANT_URL,     "❌ QDRANT_URL tidak ditemukan di .env!"
assert QDRANT_API_KEY, "❌ QDRANT_API_KEY tidak ditemukan di .env!"
assert NVIDIA_API_KEY,   "❌ NVIDIA_API_KEY tidak ditemukan di .env!"
assert NVIDIA_MODEL,   "❌ NVIDIA_MODEL tidak ditemukan di .env!"
assert NVIDIA_BASE_URL,   "❌ NVIDIA_BASE_URL tidak ditemukan di .env!"


print("✅ Konfigurasi berhasil dimuat!")
print(f"   📁 Folder MD          : {MD_FOLDER_PATH}")
print(f"   🤖 NVIDIA_MODEL Model         : {NVIDIA_MODEL}")
print(f"   ✂️  Max chars/chunk    : {MAX_CHARS_PER_CHUNK}")
print(f"   📤 Max tokens output  : {MAX_TOKENS_OUTPUT}")

✅ Konfigurasi berhasil dimuat!
   📁 Folder MD          : /kaggle/input/datasets/brianmaxwellketaren/panduan-skripsi-rag-md
   🤖 NVIDIA_MODEL Model         : meta/llama-4-maverick-17b-128e-instruct
   ✂️  Max chars/chunk    : 1500
   📤 Max tokens output  : 2048


In [3]:
import glob
import os

def load_markdown_files(folder_path: str) -> list[dict]:
    """
    Membaca semua file .md dari folder yang ditentukan.
    Returns list of dict: {filename, content}
    """
    md_files = glob.glob(os.path.join(folder_path, "**/*.md"), recursive=True)
    md_files += glob.glob(os.path.join(folder_path, "*.md"))
    md_files = list(set(md_files))  # deduplikasi

    if not md_files:
        raise FileNotFoundError(f"Tidak ada file .md ditemukan di folder: {folder_path}")

    documents = []
    for filepath in sorted(md_files):
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read().strip()
        if content:
            documents.append({
                "filename": os.path.basename(filepath),
                "filepath": filepath,
                "content": content
            })
            print(f"   📄 {os.path.basename(filepath)} ({len(content):,} karakter)")

    print(f"\n✅ Total {len(documents)} file .md berhasil dibaca!")
    return documents

documents = load_markdown_files(MD_FOLDER_PATH)

   📄 Pedoman skripsi tambahan.md (25,807 karakter)
   📄 Pedoman tulisan hanya gambar.md (14,206 karakter)
   📄 Pedoman tulisan tanpa gambar.md (61,483 karakter)
   📄 Prosedur Tugas Akhir Prodi S-1 Ilmu Komputer.md (10,154 karakter)

✅ Total 4 file .md berhasil dibaca!


# 3. Semantic Chunking

In [4]:
from semantic_text_splitter import TextSplitter
from sentence_transformers import SentenceTransformer
import numpy as np
import re

def semantic_chunk_text(
    text: str,
    embed_model: SentenceTransformer,
    window_size: int = 2,
    percentile_threshold: int = 80,
    min_sentences: int = 10
) -> list[str]:
    """
    Melakukan semantic chunking dengan pendekatan percentile breakpoint.

    Algoritma:
    1. Pisahkan teks menjadi kalimat.
    2. Gabungkan kalimat dalam window (sliding window) → group.
    3. Hitung cosine similarity antar group yang berdekatan.
    4. Hitung breakpoint pada persentil ke-N dari distribusi similarity.
    5. Potong teks di titik similarity yang rendah (di bawah threshold).
    6. Gabungkan chunk yang terlalu kecil (< min_sentences) ke chunk berikutnya.
    """
    # Tokenisasi kalimat sederhana
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    if len(sentences) <= window_size:
        return [text]  # Tidak cukup kalimat untuk di-chunk

    # Buat window groups
    groups = []
    for i in range(len(sentences)):
        start = max(0, i - window_size // 2)
        end   = min(len(sentences), i + window_size // 2 + 1)
        groups.append(" ".join(sentences[start:end]))

    # Embedding semua groups
    embeddings = embed_model.encode(groups, show_progress_bar=False, normalize_embeddings=True)

    # Cosine similarity antar group berurutan
    similarities = [
        float(np.dot(embeddings[i], embeddings[i + 1]))
        for i in range(len(embeddings) - 1)
    ]

    # Hitung breakpoint (titik dengan similarity RENDAH = topik berganti)
    distances = [1 - s for s in similarities]
    breakpoint_val = np.percentile(distances, percentile_threshold)

    # Temukan indeks breakpoint
    break_indices = [i + 1 for i, d in enumerate(distances) if d >= breakpoint_val]

    # Potong kalimat berdasarkan breakpoint → hasilkan segmen awal
    segments = []   # list of list[str] (kumpulan kalimat per segmen)
    prev_idx = 0
    for idx in break_indices:
        seg = sentences[prev_idx:idx]
        if seg:
            segments.append(seg)
        prev_idx = idx
    remaining = sentences[prev_idx:]
    if remaining:
        segments.append(remaining)

    if not segments:
        return [text]

    # ── Enforce min_sentences: gabung chunk kecil ke chunk berikutnya ────────
    merged: list[list[str]] = []
    buffer: list[str] = []

    for seg in segments:
        buffer.extend(seg)
        if len(buffer) >= min_sentences:
            merged.append(buffer)
            buffer = []

    # Sisa buffer: jika cukup besar jadikan chunk sendiri,
    # jika tidak gabungkan ke chunk terakhir
    if buffer:
        if merged and len(buffer) < min_sentences:
            merged[-1].extend(buffer)   # gabung ke chunk terakhir
        else:
            merged.append(buffer)       # jadikan chunk baru

    # Konversi list kalimat → string
    chunks = [" ".join(seg).strip() for seg in merged if seg]
    return chunks if chunks else [text]


# ─── Load embedding model ─────────────────────────────────────────────────────
print(f"⏳ Memuat embedding model: {EMBEDDING_MODEL} ...")
embed_model = SentenceTransformer(EMBEDDING_MODEL)
print(f"✅ Model berhasil dimuat! Dimensi vektor: {embed_model.get_sentence_embedding_dimension()}")


# ─── Lakukan semantic chunking untuk semua dokumen ───────────────────────────
all_chunks = []  # List of dict: {chunk_id, text, source_file}

for doc in documents:
    chunks = semantic_chunk_text(
        text=doc["content"],
        embed_model=embed_model,
        window_size=WINDOW_SIZE,
        percentile_threshold=PERCENTILE_THRESH,
        min_sentences=MIN_SENTENCES
    )
    for i, chunk in enumerate(chunks):
        sent_count = len([s for s in re.split(r'(?<=[.!?])\s+', chunk) if s.strip()])
        all_chunks.append({
            "chunk_id"       : f"{doc['filename']}_chunk_{i}",
            "text"           : chunk,
            "source"         : doc["filename"],
            "filepath"       : doc["filepath"],
            "sentence_count" : sent_count
        })
    print(f"   ✂️  {doc['filename']}: {len(chunks)} chunks "
          f"(min {min(len([s for s in re.split(r'(?<=[.!?])\s+', c) if s.strip()]) for c in chunks)} kalimat/chunk)")

print(f"\n✅ Total chunks: {len(all_chunks)}")
if all_chunks:
    counts = [c['sentence_count'] for c in all_chunks]
    print(f"   Rata-rata kalimat/chunk : {sum(counts)/len(counts):.1f}")
    print(f"   Min kalimat/chunk       : {min(counts)}")
    print(f"   Max kalimat/chunk       : {max(counts)}")
print(f"\n📋 Contoh chunk pertama:")
print("-" * 60)
print(all_chunks[0]['text'][:400] if all_chunks else '(kosong)')


⏳ Memuat embedding model: paraphrase-multilingual-MiniLM-L12-v2 ...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_125/3996098966.py:93: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"✅ Model berhasil dimuat! Dimensi vektor: {embed_model.get_sentence_embedding_dimension()}")


✅ Model berhasil dimuat! Dimensi vektor: 384
   ✂️  Pedoman skripsi tambahan.md: 50 chunks (min 2 kalimat/chunk)
   ✂️  Pedoman tulisan hanya gambar.md: 12 chunks (min 2 kalimat/chunk)
   ✂️  Pedoman tulisan tanpa gambar.md: 116 chunks (min 2 kalimat/chunk)
   ✂️  Prosedur Tugas Akhir Prodi S-1 Ilmu Komputer.md: 13 chunks (min 2 kalimat/chunk)

✅ Total chunks: 191
   Rata-rata kalimat/chunk : 5.8
   Min kalimat/chunk       : 2
   Max kalimat/chunk       : 44

📋 Contoh chunk pertama:
------------------------------------------------------------
PEDOMAN TUGAS AKHIR
PRODI S-1 ILMU KOMPUTER
JUNI
2022

No. Dokumen :
PEDOMAN TUGAS
Edisi :
AKHIR/SKRIPSI Revisi :
Tanggal Terbit : 2022
Halaman : 1 dari 14
Penetapan : Keputusan Rektor Universitas Sumatera Utara
NOMOR 2326/UN5.1.R/SK/SPB/2022
PEDOMAN PELAKSANAAN PENELITIAN PENYELESAIAN STUDI
UNIVERSITAS SUMATERA UTARA
Link : s.id/1LO6r
1. PENDAHULUAN
Program Studi Ilmu Komputer senantiasa berupaya


# 4. Vector Embedding

In [5]:
from tqdm.auto import tqdm

print(f"⏳ Melakukan embedding {len(all_chunks)} chunks ...")

texts = [c["text"] for c in all_chunks]

embeddings = embed_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True  # Cosine similarity → dot product
)

print(f"\n✅ Embedding selesai!")
print(f"   Shape: {embeddings.shape}")
print(f"   Dimensi: {embeddings.shape[1]}")

⏳ Melakukan embedding 191 chunks ...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]


✅ Embedding selesai!
   Shape: (191, 384)
   Dimensi: 384


# 5. Simpan Vektor Embedding ke Qdrant (Server Mode)

In [6]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams,
    PointStruct, PayloadSchemaType
)

# ─── Koneksi ke Qdrant Server ─────────────────────────────────────────────────
print(f"⏳ Menghubungkan ke Qdrant di {QDRANT_URL} ...")
qdrant = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=60
)
print("✅ Koneksi berhasil!")


# ─── Buat atau reset collection ───────────────────────────────────────────────
existing_collections = [c.name for c in qdrant.get_collections().collections]

if COLLECTION_NAME in existing_collections:
    print(f"⚠️  Collection '{COLLECTION_NAME}' sudah ada, menghapus dan membuat ulang ...")
    qdrant.delete_collection(COLLECTION_NAME)

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=VECTOR_DIM,
        distance=Distance.COSINE
    )
)
print(f"✅ Collection '{COLLECTION_NAME}' berhasil dibuat!")


# ─── Upsert semua vektor ──────────────────────────────────────────────────────
BATCH_SIZE = 100

points = [
    PointStruct(
        id=idx,
        vector=embeddings[idx].tolist(),
        payload={
            "chunk_id" : all_chunks[idx]["chunk_id"],
            "text"     : all_chunks[idx]["text"],
            "source"   : all_chunks[idx]["source"],
            "filepath" : all_chunks[idx]["filepath"]
        }
    )
    for idx in range(len(all_chunks))
]

print(f"\n⏳ Menyimpan {len(points)} vektor ke Qdrant (batch size={BATCH_SIZE}) ...")

for i in tqdm(range(0, len(points), BATCH_SIZE), desc="Upload"):
    batch = points[i:i + BATCH_SIZE]
    qdrant.upsert(collection_name=COLLECTION_NAME, points=batch)

# Verifikasi
count = qdrant.count(collection_name=COLLECTION_NAME).count
print(f"\n✅ Upload selesai! Total vektor tersimpan: {count}")

⏳ Menghubungkan ke Qdrant di https://ea295c4d-3efb-4676-9964-a20d09b4e162.us-east4-0.gcp.cloud.qdrant.io ...
✅ Koneksi berhasil!
⚠️  Collection 'Thesis Guide for USU Computer Science Students' sudah ada, menghapus dan membuat ulang ...
✅ Collection 'Thesis Guide for USU Computer Science Students' berhasil dibuat!

⏳ Menyimpan 191 vektor ke Qdrant (batch size=100) ...


Upload:   0%|          | 0/2 [00:00<?, ?it/s]


✅ Upload selesai! Total vektor tersimpan: 191


# 6. Input Query dari User

In [7]:
# ─── Masukkan pertanyaan di sini ──────────────────────────────────────────────
user_query = "Apa langkah-langkah maju seminar proposal?"

if not user_query:
    raise ValueError("❌ Query tidak boleh kosong!")

print(f"\n📝 Query: {user_query}")


📝 Query: Apa langkah-langkah maju seminar proposal?


# 7. Embedding Query User

In [8]:
print("⏳ Melakukan embedding query ...")

query_embedding = embed_model.encode(
    user_query,
    normalize_embeddings=True
).tolist()

print(f"✅ Query berhasil di-embed! Dimensi: {len(query_embedding)}")

⏳ Melakukan embedding query ...
✅ Query berhasil di-embed! Dimensi: 384


# 8. Retrieval dari Qdrant (Top-K = 25)

In [9]:
print(f"⏳ Melakukan retrieval top-{TOP_K} dari Qdrant ...")

# qdrant-client v2.x: gunakan query_points (pengganti .search yang deprecated)
search_results = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    query=query_embedding,
    limit=TOP_K,
    with_payload=True
).points

retrieved_chunks = [
    {
        "id"     : hit.id,
        "text"   : hit.payload["text"],
        "source" : hit.payload["source"],
        "score"  : hit.score
    }
    for hit in search_results
]

print(f"✅ Berhasil retrieve {len(retrieved_chunks)} chunks!")
print("\n📋 Top 5 hasil retrieval (sebelum reranking):")
print("-" * 60)
for i, chunk in enumerate(retrieved_chunks[:5], 1):
    print(f"[{i}] Score: {chunk['score']:.4f} | Sumber: {chunk['source']}")
    print(f"    {chunk['text'][:150]}...\n")

⏳ Melakukan retrieval top-25 dari Qdrant ...
✅ Berhasil retrieve 25 chunks!

📋 Top 5 hasil retrieval (sebelum reranking):
------------------------------------------------------------
[1] Score: 0.7561 | Sumber: Pedoman skripsi tambahan.md
    4. Tahap Pelaksanaan Seminar Proposal
• Penjadwalan: Pihak Prodi / Fakultas melakukan Penjadwalan Seminar Proposal. • Pelaksanaan: Prodi / Fakultas me...

[2] Score: 0.7269 | Sumber: Pedoman skripsi tambahan.md
    6. Telah melakukan bimbingan proposal dengan kedua dosen pembimbing dan proposal
sudah di-acc oleh kedua dosen pembimbing. b. Seminar Proposal
Pada pe...

[3] Score: 0.6586 | Sumber: Pedoman skripsi tambahan.md
    3. Mahasiswa wajib menyiapkan diri untuk memberikan penjelasan mengenai proposal yang
diajukan pada seminar proposal. 4. Mahasiswa akan memaparkan ren...

[4] Score: 0.6533 | Sumber: Prosedur Tugas Akhir Prodi S-1 Ilmu Komputer.md
    Prosedur Tugas Akhir Prodi S-1 Ilmu Komputer

Diperuntukkan Bagi Mahasiswa Program Studi Ilm

# 9. Reranking dengan CrossEncoder (Top-5)

In [10]:
from sentence_transformers import CrossEncoder

RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-12-v2"

# ─── Inisialisasi CrossEncoder Reranker ──────────────────────────────────────
print(f"⏳ Memuat CrossEncoder reranker: {RERANKER_MODEL} ...")
cross_encoder = CrossEncoder(
    model_name=RERANKER_MODEL,
    max_length=512
)
print("✅ CrossEncoder reranker siap!")

The CrossEncoder `model_name` argument was renamed and is now deprecated. Please use `model_name_or_path` instead.


⏳ Memuat CrossEncoder reranker: cross-encoder/ms-marco-MiniLM-L-12-v2 ...


config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ CrossEncoder reranker siap!


In [11]:
# ─── Buat pasangan (query, passage) untuk scoring ────────────────────────────
pairs = [[user_query, chunk["text"]] for chunk in retrieved_chunks]

# ─── Lakukan reranking ────────────────────────────────────────────────────────
print(f"\n⏳ Melakukan reranking {len(pairs)} chunks ...")
rerank_scores = cross_encoder.predict(pairs, show_progress_bar=True)

# Gabungkan score ke retrieved_chunks, lalu sort descending
for chunk, score in zip(retrieved_chunks, rerank_scores):
    chunk["rerank_score"] = float(score)

top_reranked = sorted(
    retrieved_chunks,
    key=lambda x: x["rerank_score"],
    reverse=True
)[:TOP_N_RERANK]

print(f"\n✅ Reranking selesai! Menampilkan top-{TOP_N_RERANK}:")
print("-" * 60)
for idx, result in enumerate(top_reranked, 1):
    print(f"[{idx}] Rerank Score : {result['rerank_score']:.4f} | Sumber: {result['source']}")
    print(f"     Vector Score: {result['score']:.4f}")
    print(f"     {result['text'][:150]}...\n")


⏳ Melakukan reranking 25 chunks ...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Reranking selesai! Menampilkan top-10:
------------------------------------------------------------
[1] Rerank Score : 2.1155 | Sumber: Prosedur Tugas Akhir Prodi S-1 Ilmu Komputer.md
     Vector Score: 0.6527
     Penelitian
9. Exum (Judul, Latar Belakang, Rumusan Masalah, Metodologi, Referensi)
10. Tanda tangan mahasiswa

Seminar Proposal

Untuk  maju  seminar...

[2] Rerank Score : 1.0931 | Sumber: Pedoman skripsi tambahan.md
     Vector Score: 0.7269
     6. Telah melakukan bimbingan proposal dengan kedua dosen pembimbing dan proposal
sudah di-acc oleh kedua dosen pembimbing. b. Seminar Proposal
Pada pe...

[3] Rerank Score : 0.2948 | Sumber: Pedoman skripsi tambahan.md
     Vector Score: 0.7561
     4. Tahap Pelaksanaan Seminar Proposal
• Penjadwalan: Pihak Prodi / Fakultas melakukan Penjadwalan Seminar Proposal. • Pelaksanaan: Prodi / Fakultas me...

[4] Rerank Score : -0.4444 | Sumber: Pedoman skripsi tambahan.md
     Vector Score: 0.6586
     3. Mahasiswa wajib menyiapkan d

# 10. Mengatur Final Prompt (Vectors + Query + Prompt Template)

In [12]:
# ─── Gabungkan context dari top reranked chunks ───────────────────────────────
context_parts = []
for idx, result in enumerate(top_reranked, 1):
    source = result["source"]
    text   = result["text"]
    context_parts.append(f"[Source {idx}: {source}]\n{text}")

context = "\n\n".join(context_parts)

# ─── Prompt Template ─────────────────────────────────────────────────────────
PROMPT_TEMPLATE = """Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context:
{context}

Query: {query}

Answer the question and provide additional helpful information,
based on the pieces of information, if applicable. Be succinct.
Responses should be properly formatted to be easily read."""

final_prompt = PROMPT_TEMPLATE.format(
    context=context,
    query=user_query
)

print("✅ Prompt berhasil dibuat!")
print(f"   Total karakter prompt: {len(final_prompt):,}")
print("\n" + "=" * 60)
print("PREVIEW PROMPT:")
print("=" * 60)
print(final_prompt[:800] + "\n...[dipotong untuk preview]" if len(final_prompt) > 800 else final_prompt)

✅ Prompt berhasil dibuat!
   Total karakter prompt: 17,198

PREVIEW PROMPT:
Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context:
[Source 1: Prosedur Tugas Akhir Prodi S-1 Ilmu Komputer.md]
Penelitian
9. Exum (Judul, Latar Belakang, Rumusan Masalah, Metodologi, Referensi)
10. Tanda tangan mahasiswa

Seminar Proposal

Untuk  maju  seminar  proposal  mahasiswa  perlu  menyelesaikan  dua  submission  di  bawah  ini
secara berurutan. 1. Persetujuan Seminar Proposal

Mahasiswa  diharuskan  mendapatkan  persetujuan  untuk  mengikuti  seminar  Proposal  oleh
dosen pembimbing melalui modul ini. Mahasiswa wajib mengirimkan file submission berupa :

1. File Proposal dengan format :  PROPOSAL_NIM.pdf
2. File  Form  Persetujuan  Seminar  Proposal  yang  suda
...[dipotong untuk preview]


# 11. Generate Hasil Menggunakan LLM (Llama 4)

In [13]:
from openai import OpenAI

# ─── Inisialisasi NVIDIA NIM client ──────────────────────────────────────────
nim_client = OpenAI(
    api_key=NVIDIA_API_KEY,
    base_url=NVIDIA_BASE_URL  # contoh: "https://integrate.api.nvidia.com/v1"
)

print(f"⏳ Mengirim prompt ke NVIDIA NIM ({NVIDIA_MODEL}) ...")

chat_completion = nim_client.chat.completions.create(
    model=NVIDIA_MODEL,  # contoh: "meta/llama-3.1-8b-instruct"
    messages=[
        {
            "role"   : "system",
            "content": "You are a helpful assistant that answers questions based on provided context."
        },
        {
            "role"   : "user",
            "content": final_prompt
        }
    ],
    temperature=0.2,
    max_tokens=1024,
    top_p=0.9
)

llm_answer = chat_completion.choices[0].message.content
usage      = chat_completion.usage

print("✅ Respons diterima dari NVIDIA NIM!")
print(f"   Prompt tokens     : {usage.prompt_tokens:,}")
print(f"   Completion tokens : {usage.completion_tokens:,}")
print(f"   Total tokens      : {usage.total_tokens:,}")

from IPython.display import Markdown, display
print("=" * 70)
print("🔍 PERTANYAAN:")
print("=" * 70)
print(user_query)
print("\n" + "=" * 70)
print(f"🤖 JAWABAN ({NVIDIA_MODEL}):")
print("=" * 70)
display(Markdown(llm_answer))

⏳ Mengirim prompt ke NVIDIA NIM (meta/llama-4-maverick-17b-128e-instruct) ...
✅ Respons diterima dari NVIDIA NIM!
   Prompt tokens     : 4,261
   Completion tokens : 236
   Total tokens      : 4,497
🔍 PERTANYAAN:
Apa langkah-langkah maju seminar proposal?

🤖 JAWABAN (meta/llama-4-maverick-17b-128e-instruct):


Untuk maju seminar proposal, mahasiswa perlu menyelesaikan dua submission secara berurutan, yaitu:

1. **Persetujuan Seminar Proposal**: 
 - Mengirimkan file proposal dengan format: PROPOSAL_NIM.pdf
 - Mengirimkan file Form Persetujuan Seminar Proposal yang sudah diisi dan ditandatangani oleh Dosen Pembimbing dengan format: DOPING_NIM.pdf, beserta lembar kendali bimbingan pra proposal yang telah ditandatangani oleh dosen pembimbing.

2. **Workshop Seminar Proposal**: 
 - Melakukan submission kembali dengan proposal yang sudah siap dengan format: Proposal_NIM.pdf

Sebelum melakukan submission, mahasiswa juga harus memenuhi persyaratan lainnya, seperti:
- Telah melakukan bimbingan proposal dengan kedua dosen pembimbing dan proposal sudah di-acc oleh kedua dosen pembimbing.
- Exum (Judul, Latar Belakang, Rumusan Masalah, Metodologi, Referensi) telah di-acc oleh kedua dosen pembimbing.

Setelah seminar proposal, jika proposal disetujui dengan melakukan perbaikan, mahasiswa harus melakukan perbaikan yang diminta paling lambat 1 minggu setelah seminar proposal dan menyerahkan perbaikan tersebut kepada masing-masing pembimbing dan pembanding.

# 12. Mode Interaktif — Tanya Berulang

In [14]:
def ask(query: str) -> str:
    """Fungsi lengkap: query → embed → retrieve → rerank → generate → return answer."""

    # Embed query
    q_embedding = embed_model.encode(query, normalize_embeddings=True).tolist()

    # Retrieve top-K dari Qdrant
    hits = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=q_embedding,
        limit=TOP_K,
        with_payload=True
    ).points

    candidates = [
        {
            "id"     : h.id,
            "text"   : h.payload["text"],
            "source" : h.payload["source"],
            "score"  : h.score
        }
        for h in hits
    ]

    # CrossEncoder reranking
    pairs  = [[query, c["text"]] for c in candidates]
    scores = cross_encoder.predict(pairs)
    for c, s in zip(candidates, scores):
        c["rerank_score"] = float(s)

    reranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)[:TOP_N_RERANK]

    # Build context
    context = "\n\n".join(
        f"[Source {i+1}: {r['source']}]\n{r['text']}"
        for i, r in enumerate(reranked)
    )

    # Build prompt
    prompt = PROMPT_TEMPLATE.format(context=context, query=query)

    resp = nim_client.chat.completions.create(
        model=NVIDIA_MODEL,  # contoh: "meta/llama-3.1-8b-instruct"
        messages=[
            {
                "role"   : "system",
                "content": "You are a helpful assistant that answers questions based on provided context."
            },
            {
                "role"   : "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=1024,
        top_p=0.9
    )
    return resp.choices[0].message.content
    
# ─── Loop interaktif ──────────────────────────────────────────────────────────
print("💬 Mode Interaktif RAG — ketik 'keluar' untuk berhenti\n")
while True:
    q = input("🔍 Pertanyaan: ").strip()
    if q.lower() in ("keluar", "exit", "quit", "q"):
        print("👋 Sesi selesai.")
        break
    if not q:
        continue
    print("\n⏳ Memproses ...\n")
    answer = ask(q)
    from IPython.display import Markdown, display
    
    print("=" * 70)
    print("🔍 PERTANYAAN:")
    print("=" * 70)
    print(q)
    
    print("\n" + "=" * 70)
    print(f"🤖 JAWABAN  ({NVIDIA_MODEL}):")
    print("=" * 70)
    
    display(Markdown(answer))

💬 Mode Interaktif RAG — ketik 'keluar' untuk berhenti



🔍 Pertanyaan:  apa persyaratan maju sidang meja hijau?



⏳ Memproses ...

🔍 PERTANYAAN:
apa persyaratan maju sidang meja hijau?

🤖 JAWABAN  (meta/llama-4-maverick-17b-128e-instruct):


Untuk maju Sidang Meja Hijau, mahasiswa harus memenuhi beberapa persyaratan. Berdasarkan informasi yang tersedia, persyaratan tersebut adalah:

1. Mahasiswa harus sudah mengambil Mata Kuliah Tugas Akhir (MK TA) di KRS pada semester berjalan.
2. Mahasiswa wajib mengirimkan file submission berupa:
 * File Proposal dengan format: SKRIPSI_NIM.pdf
 * File Persetujuan Sidang Meja Hijau yang telah diisi Nama Dosen Pembimbing 1 & 2, Nama Dosen Pembanding 1 & 2 dengan format: PERSETUJUAN_NIM.pdf dan wajib ditandatangani oleh dosen pembimbing dan dosen pembanding.
 * Form lembar kendali bimbingan pra sidang meja hijau yang telah ditandatangani oleh Dosen Pembimbing 1 & 2.
3. Dosen Pembimbing 1 melakukan approval pada submission ini untuk dapat memberikan izin melanjutkan ke Sidang Meja Hijau. Approval ditandai dengan kriteria Nilai >= 50 (Disetujui) dan < 50 (Tidak Disetujui).

Dengan demikian, mahasiswa harus memenuhi persyaratan administratif dan mendapatkan persetujuan dari dosen pembimbing untuk dapat maju ke Sidang Meja Hijau.

🔍 Pertanyaan:  apa persyaratan maju seminar hasil?



⏳ Memproses ...

🔍 PERTANYAAN:
apa persyaratan maju seminar hasil?

🤖 JAWABAN  (meta/llama-4-maverick-17b-128e-instruct):


Untuk maju seminar hasil, mahasiswa perlu menyelesaikan beberapa submission secara berurutan. Berdasarkan Source 5: Prosedur Tugas Akhir Prodi S-1 Ilmu Komputer.md, ada enam submission yang harus diselesaikan, namun informasi lebih detail diberikan untuk empat submission awal.

Berikut adalah persyaratan maju seminar hasil:

1. **Persetujuan Seminar Hasil**: 
 - Mengirimkan file submission berupa File Proposal dengan format: SKRIPSI_NIM.pdf
 - Mengirimkan File Persetujuan Seminar Hasil yang telah diisi dan ditandatangani oleh Dosen Pembimbing 1 & 2 dengan format: PERSETUJUAN_NIM.pdf
 - Mengirimkan Form lembar kendali bimbingan pra seminar hasil yang telah ditandatangani oleh Dosen Pembimbing 1 & 2.

2. **Dosen Pembimbing 1 melakukan approval** pada submission mahasiswa untuk memberikan izin melanjutkan ke Seminar Hasil dengan nilai >= 50.

3. **Berkas Berita Acara Seminar Hasil**: Mengirimkan file submission berupa File Berita Acara Seminar Hasil dengan format: BASemHas_NIM.doc/docx.

4. **Workshop Seminar Hasil**: Melakukan submission kembali dengan skripsi yang sudah siap dengan format: SKRIPSI_NIM.pdf.

Selain itu, berdasarkan Source 4: Pedoman skripsi tambahan.md, mahasiswa harus sudah mengambil Mata Kuliah Tugas Akhir (TA) di KRS pada semester berjalan untuk dapat melaksanakan Seminar Hasil.

Dengan demikian, persyaratan maju seminar hasil meliputi beberapa submission dan pengambilan Mata Kuliah TA di KRS semester berjalan.

🔍 Pertanyaan:  exit


👋 Sesi selesai.


In [15]:
import os, json, time, re
from datetime import datetime
from typing import List, Dict, Any

try:
    from ragas import evaluate, RunConfig
    from ragas.metrics import (
        Faithfulness,
        AnswerRelevancy,
        ContextPrecision,
        ContextRecall,
        AnswerCorrectness,
        AnswerSimilarity
    )
    from ragas.llms       import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from datasets import Dataset
    print("✅ RAGAS berhasil diimpor")
except ImportError as e:
    raise ImportError(f"Import gagal: {e}\n Jalankan: !pip install -q ragas datasets langchain-groq langchain-community")

✅ RAGAS berhasil diimpor


In [16]:
EVAL_QUESTIONS: List[str] = [
    # === Bagian 1: Kegiatan Seminar dan Sidang ===
    "Berapa batas minimal dan maksimal jumlah peserta dalam satu kali kegiatan seminar dan sidang?",
    "Jika mahasiswa mengikuti Seminar Proposal pada tanggal 14 April 2026, kapan batas akhir pengumpulan berkasnya?",
    "Sebutkan format penamaan file untuk Proposal dan Form Persetujuan Seminar Proposal yang harus diunggah pada tahap Persetujuan Seminar Proposal!",
    "Apa format nama file submission dan apa saja yang harus disertakan setelah mahasiswa selesai melakukan uji program secara online?",


    # === Bagian 2: Format Penulisan Skripsi ===
    "Berapa ukuran, jenis, dan warna kertas yang diwajibkan untuk salinan naskah asli skripsi?",
    "Apa perbedaan warna kulit depan skripsi antara Program Studi S1 Ilmu Komputer dan Program Studi S1 Teknologi Informasi?",
    "Berapa spesifikasi margin (atas, kanan, kiri, bawah) yang harus digunakan pada halaman awal setiap bab?",
    "Bagaimana aturan penomoran halaman untuk bagian muka skripsi dibandingkan dengan bagian tubuh skripsi?",


    # === Bagian 3: Lampiran dan Detail Teknis ===
    "Berapa ukuran margin tepi kertas yang ditunjukkan pada Lampiran 3 untuk penulisan awal bab?",
    "Apa saja informasi teks yang terdapat pada tulang belakang skripsi S1 Ilmu Komputer berdasarkan Lampiran 2A?",
    "Berdasarkan Lampiran 4A, kalimat apa yang tertulis tepat di bawah tulisan 'SKRIPSI' pada halaman judul skripsi S1 Ilmu Komputer?",
    "Berapa spasi yang digunakan untuk penulisan sub-bab '1.1. Latar Belakang' menurut Lampiran 3?",

    # === Bagian 4: Executive Summary dan Persyaratan ===
    "Apa yang dimaksud dengan Executive Summary (Exum) dan komponen apa saja yang harus ada di dalamnya menurut pedoman?",
    "Bagaimana kriteria penilaian kelayakan Executive Summary (Exum) oleh Kepala Laboratorium dan apa konsekuensi dari masing-masing nilai tersebut?",
    "Apa saja persyaratan akademik dan administratif yang harus dipenuhi mahasiswa untuk dapat mengajukan Seminar Proposal Tugas Akhir?",
    "Bagaimana ketentuan pakaian (dress code) yang wajib dikenakan oleh mahasiswa saat pelaksanaan Seminar Proposal?",

]

GROUND_TRUTHS: List[str] = [
    # === Bagian 1: Kegiatan Seminar dan Sidang ===
    "Jumlah peserta minimal 1 orang dan maksimal 7 orang dalam satu kali kegiatan seminar dan sidang.",
    "Batas akhir pengumpulan berkas adalah pada 08 April 2026 Pukul 11.00 WIB.",
    "File Proposal dengan format PROPOSAL_NIM.pdf dan File Form Persetujuan Seminar Proposal dengan format DOPING_NIM.pdf.",
    "Format file submission adalah NIM_Judul Tugas akhir dan wajib menyertakan Link Video Demo Program.",

    # === Bagian 2: Format Penulisan Skripsi ===
    "Ukuran kertas yang diwajibkan adalah A4 (210 mm x 297 mm). Jenis kertas untuk naskah asli adalah HVS dengan berat 70 gram (mg), dan warnanya harus putih.",
    "Program Studi S1 Ilmu Komputer menggunakan warna kulit Hitam, sedangkan Program Studi S1 Teknologi Informasi menggunakan warna kulit Abu-abu.",
    "Spesifikasi margin awal bab adalah: Margin atas 50 mm, Margin kanan 25 mm, Margin kiri 38 mm, dan Margin bawah 25 mm dari tepi kertas.",
    "Bagian muka skripsi menggunakan nomor halaman dengan angka Romawi kecil (i, ii, iii, ...), sedangkan bagian tubuh skripsi menggunakan angka Arab (1, 2, 3, ...).",

    # === Bagian 3: Lampiran dan Detail Teknis ===
    "Margin tepi kertas pada Lampiran 3 menunjukkan ukuran 50 mm, 38 mm, dan 25 mm.",
    "Informasi pada tulang belakang skripsi S1 Ilmu Komputer meliputi: NAMA MAHASISWA, S1 ILMU KOMPUTER, dan Fasilkom-TI 2012.",
    "Kalimat yang tertulis adalah: 'Diajukan untuk melengkapi tugas dan memenuhi syarat memperoleh ijazah Sarjana Ilmu Komputer'.",
    "Penulisan sub-bab '1.1. Latar Belakang' menggunakan 4 spasi.",


    # === Bagian 4: Executive Summary dan Persyaratan ===
    "Executive Summary (Exum) adalah pengajuan ide tugas akhir yang diajukan mahasiswa sebelum mengajukan proposal. Komponen Exum terdiri dari Judul, Latar Belakang, Rumusan Masalah, Metodologi, dan Referensi.",
    "Penilaian kelayakan Exum oleh Kepala Laboratorium berupa 'layak tanpa perbaikan' (nilai 100), 'layak dengan perbaikan' (70 <= nilai <= 99), dan 'tidak layak' (nilai < 70). Jika layak, mahasiswa lanjut ke tahap proposal; jika dengan perbaikan, mahasiswa harus merevisi sesuai catatan; jika tidak layak, mahasiswa harus melakukan pergantian judul.",
    "Persyaratan untuk mengajukan Seminar Proposal antara lain: telah menyelesaikan minimal 100 SKS dengan IPK minimal 2,00; telah menyelesaikan PKL atau program MBKM; telah menyelesaikan semua mata kuliah wajib; telah lulus mata kuliah Metodologi Penelitian; serta proposal sudah di-acc oleh kedua dosen pembimbing.",
    "Mahasiswa wajib menggunakan atasan putih, blazer hitam, dan bawahan hitam. Khusus untuk mahasiswa laki-laki wajib menggunakan dasi."
]

assert len(EVAL_QUESTIONS) == len(GROUND_TRUTHS), \
    f"❌ Jumlah pertanyaan ({len(EVAL_QUESTIONS)}) ≠ ground truth ({len(GROUND_TRUTHS)})"

In [17]:
from langchain_openai import ChatOpenAI

In [18]:
def setup_ragas_judge():
    # ── Judge LLM: NVIDIA NIM ──
    judge_llm = LangchainLLMWrapper(
        ChatOpenAI(
            model=NVIDIA_MODEL,
            openai_api_key=NVIDIA_API_KEY,
            openai_api_base=NVIDIA_BASE_URL,
            temperature=0.01,
            max_tokens=2048,
            n=1,
        )
    )
    # ── Judge Embedding: HuggingFace lokal ──
    from langchain_community.embeddings import HuggingFaceEmbeddings
    judge_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(
        model_name="paraphrase-multilingual-MiniLM-L12-v2",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    ))
    return judge_llm, judge_emb

In [19]:
def ask_with_context(query: str):
    """
    Sama persis dengan ask() di notebook, tapi juga mengembalikan list contexts
    (dibutuhkan RAGAS untuk menghitung context_precision & context_recall).

    Returns:
        answer   (str)        : jawaban LLM
        contexts (list[str])  : list teks chunk yang masuk ke prompt
    """
    # Embed query
    q_embedding = embed_model.encode(query, normalize_embeddings=True).tolist()

    # Retrieve top-K dari Qdrant
    hits = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=q_embedding,
        limit=TOP_K,
        with_payload=True
    ).points

    candidates = [
        {
            "id"     : h.id,
            "text"   : h.payload["text"],
            "source" : h.payload["source"],
            "score"  : h.score
        }
        for h in hits
    ]

    # CrossEncoder reranking
    pairs         = [[query, c["text"]] for c in candidates]
    rerank_scores = cross_encoder.predict(pairs)
    for c, s in zip(candidates, rerank_scores):
        c["rerank_score"] = float(s)

    # Ambil TOP_N_RERANK setelah reranking
    top_reranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)[:TOP_N_RERANK]

    # Contexts sebagai list string (format RAGAS)
    contexts = [r["text"] for r in top_reranked]

    # Potong sesuai MAX_CHARS_PER_CHUNK (sesuai notebook)
    contexts_trimmed = [c[:MAX_CHARS_PER_CHUNK] for c in contexts]

    # Build context string untuk prompt
    context_str = "\n\n".join(
        f"[Source {i+1}: {r['source']}]\n{r['text'][:MAX_CHARS_PER_CHUNK]}"
        for i, r in enumerate(top_reranked)
    )

    # Build prompt (sama dengan notebook)
    prompt = PROMPT_TEMPLATE.format(context=context_str, query=query)

    # Panggil Groq
    resp = nim_client.chat.completions.create(
        model=NVIDIA_MODEL,  # contoh: "meta/llama-3.1-8b-instruct"
        messages=[
            {
                "role"   : "system",
                "content": "You are a helpful assistant that answers questions based on provided context."
            },
            {
                "role"   : "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=MAX_TOKENS_OUTPUT,
        top_p=0.9
    )
    answer = resp.choices[0].message.content
    return answer, contexts_trimmed

In [20]:
def collect_rag_outputs(questions: List[str], ground_truths: List[str]) -> Dict[str, List]:
    """Jalankan ask_with_context() untuk setiap pertanyaan evaluasi."""
    all_questions, all_answers, all_contexts, all_gt = [], [], [], []

    print(f"📥 Mengumpulkan output RAG untuk {len(questions)} pertanyaan...\n")

    for i, (question, gt) in enumerate(zip(questions, ground_truths), 1):
        print(f"  [{i}/{len(questions)}] {question[:70]}...")
        try:
            answer, contexts = ask_with_context(question)
            all_questions.append(question)
            all_answers.append(answer)
            all_contexts.append(contexts)
            all_gt.append(gt)
            print(f"         ✅ {len(contexts)} konteks | Jawaban: {answer[:80]}...")
        except Exception as e:
            print(f"         ❌ Error: {e}")
            all_questions.append(question)
            all_answers.append("ERROR")
            all_contexts.append([""])
            all_gt.append(gt)

        time.sleep(1.0)   # hindari rate limit 
        print()

    return {
        "question"     : all_questions,
        "answer"       : all_answers,
        "contexts"     : all_contexts,
        "ground_truth" : all_gt,
    }


In [21]:
from typing import Dict, List, Any

In [22]:
def run_ragas_evaluation(rag_data: Dict[str, List]) -> Dict[str, Any]:
    """Jalankan evaluasi RAGAS — kembalikan hasil mentah + DataFrame skor."""
    print("\n" + "="*70)
    print("🧪 MEMULAI EVALUASI RAGAS")
    print("="*70)

    # Setup judge
    print("\n⚙️  Menyiapkan judge LLM & embeddings...")
    judge_llm, judge_emb = setup_ragas_judge()

    # Daftar metrik
    answer_similarity = AnswerSimilarity()
    answer_similarity.embeddings = judge_emb
    
    answer_correctness = AnswerCorrectness()
    answer_correctness.llm = judge_llm
    answer_correctness.embeddings = judge_emb
    answer_correctness.answer_similarity = answer_similarity  # ← inject manual
    
    metrics = [
        Faithfulness(),
        AnswerRelevancy(),
        ContextPrecision(),
        ContextRecall(),
        answer_correctness,  # ← pakai instance yang sudah dikonfigurasi
    ]
    
    for m in metrics:
        m.llm = judge_llm
        if hasattr(m, "embeddings"):
            m.embeddings = judge_emb

    # Buat dataset
    dataset = Dataset.from_dict(rag_data)
    print(f"\n📊 Dataset  : {len(dataset)} sampel")
    print(f"📐 Metrik   : {[type(m).__name__ for m in metrics]}\n")

    run_config = RunConfig(
        timeout=120,
        max_retries=3,
        max_wait=60,
        max_workers=1,
    )

    print("⏳ Menjalankan evaluasi...\n")
    t0 = time.time()
    result = evaluate(
        dataset=dataset,
        metrics=metrics,
        run_config=run_config,
    )
    elapsed = time.time() - t0
    print(f"\n✅ Evaluasi selesai dalam {elapsed:.1f} detik")

    scores_df = result.to_pandas()
    metric_names = [type(m).__name__ for m in metrics]
    return {
        "result"      : result,
        "scores_df"   : scores_df,
        "metric_names": metric_names,
        "elapsed_sec" : round(elapsed, 1),
        "n_samples"   : len(dataset),
    }

In [32]:
def compute_and_save_ragas_scores(eval_output: Dict[str, Any]) -> Dict[str, Any]:
    """
    Hitung skor agregat dari hasil evaluasi RAGAS.
    Kompatibel dengan RAGAS v0.1 (kolom 'question') dan v0.2+ (kolom 'user_input').
    """
    scores_df    = eval_output["scores_df"]
    print("faithfulness =",scores_df["faithfulness"].mean())
    print("answer_relevancy =",scores_df["answer_relevancy"].mean())
    print("context_precision =",scores_df["context_precision"].mean())
    print("context_recall =",scores_df["context_recall"].mean())
    print("answer_correctness =",scores_df["answer_correctness"].mean())
    
    csv_path  = f"ragas_hasil.csv"
    
    scores_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    print(f"\n💾 Hasil disimpan:")
    print(f"   CSV  : {csv_path}")

    

In [24]:
def _label(score: float) -> str:
    if score >= 0.85: return "🟢 Sangat Baik"
    if score >= 0.70: return "🟡 Baik"
    if score >= 0.55: return "🟠 Cukup"
    return  

In [25]:
# Step 1: Kumpulkan output pipeline
rag_data = collect_rag_outputs(EVAL_QUESTIONS, GROUND_TRUTHS)

# Step 2: Evaluasi RAGAS


📥 Mengumpulkan output RAG untuk 16 pertanyaan...

  [1/16] Berapa batas minimal dan maksimal jumlah peserta dalam satu kali kegia...
         ✅ 10 konteks | Jawaban: Batas minimal dan maksimal jumlah peserta dalam satu kali kegiatan seminar dan s...

  [2/16] Jika mahasiswa mengikuti Seminar Proposal pada tanggal 14 April 2026, ...
         ✅ 10 konteks | Jawaban: To determine the batas akhir pengumpulan berkas (deadline for submitting files) ...

  [3/16] Sebutkan format penamaan file untuk Proposal dan Form Persetujuan Semi...
         ✅ 10 konteks | Jawaban: Format penamaan file untuk Proposal dan Form Persetujuan Seminar Proposal yang h...

  [4/16] Apa format nama file submission dan apa saja yang harus disertakan set...
         ✅ 10 konteks | Jawaban: Setelah mahasiswa selesai melakukan uji program secara online, mahasiswa diwajib...

  [5/16] Berapa ukuran, jenis, dan warna kertas yang diwajibkan untuk salinan n...
         ✅ 10 konteks | Jawaban: Untuk salinan naskah asli skri

In [26]:
eval_output = run_ragas_evaluation(rag_data)


🧪 MEMULAI EVALUASI RAGAS

⚙️  Menyiapkan judge LLM & embeddings...


/tmp/ipykernel_125/1835101334.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  judge_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



📊 Dataset  : 16 sampel
📐 Metrik   : ['Faithfulness', 'AnswerRelevancy', 'ContextPrecision', 'ContextRecall', 'AnswerCorrectness']

⏳ Menjalankan evaluasi...



Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

Failed to parse output. Returning None.
No statements were generated from the answer.



✅ Evaluasi selesai dalam 451.9 detik


In [33]:
compute_and_save_ragas_scores(eval_output)

faithfulness = 0.8565476190476191
answer_relevancy = 0.6590425449823064
context_precision = 0.4381944444028125
context_recall = 0.6916666666666667
answer_correctness = 0.35444648696195385

💾 Hasil disimpan:
   CSV  : ragas_hasil.csv
